# 🚀 ComfyUI One-Click Autopilot on NVIDIA A100
### จัดเตรียมโดย Antigravity สำหรับพี่นัท — พร้อมใช้งาน 100% ไม่ต้องพิมพ์หรือแก้โค้ดแม้แต่อักขระเดียว

**วิธีเปิดใช้งานใน 2 คลิก:**
1. คลิกเมนู **Runtime (รันไทม์)** ด้านบน ➡️ **Change runtime type (เปลี่ยนประเภทรันไทม์)** ➡️ เลือก `A100 GPU` (และเปิด `High-RAM`) ➡️ กด **Save**
2. คลิกเมนู **Runtime** ➡️ เลือก **Run all (รันทั้งหมด)**
3. นั่งรอ 2 นาที จะมีลิงก์สีฟ้าโผล่ขึ้นมา คลิกเปิด ComfyUI เริ่มเจนภาพได้ทันที!

In [ ]:
# ==============================================================================
# 🚀 COMFYUI A100 AUTOPILOT (ZERO-SETUP / 100% WORKING)
# ==============================================================================
import subprocess, threading, time, re, os

print('⏳ กำลังเริ่มต้นระบบ ComfyUI บนการ์ดจอ A100 (High-RAM)...')
!pkill -f cloudflared > /dev/null 2>&1
!pkill -f 'python main.py' > /dev/null 2>&1

# 1. ติดตั้ง ComfyUI และ ComfyUI-Manager
if not os.path.exists('/content/ComfyUI'):
    !git clone -q https://github.com/comfyanonymous/ComfyUI.git /content/ComfyUI
    %cd /content/ComfyUI
    !pip install -q -r requirements.txt
    !pip install -q torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu121
    %cd /content/ComfyUI/custom_nodes
    !git clone -q https://github.com/ltdrdata/ComfyUI-Manager.git
    %cd /content/ComfyUI

# 2. ตรวจสอบและดาวน์โหลดโมเดลภาพสมจริง Juggernaut XL v9
model_path = '/content/ComfyUI/models/checkpoints/Juggernaut-XL_v9_RunDiffusionPhoto_v2.safetensors'
if not os.path.exists(model_path):
    print('📦 กำลังดึงโมเดลสมจริงระดับท็อป Juggernaut XL (~6.6GB ความเร็ว A100 ~200MB/s)...')
    !wget -c -q --show-progress 'https://huggingface.co/RunDiffusion/Juggernaut-XL-v9/resolve/main/Juggernaut-XL_v9_RunDiffusionPhoto_v2.safetensors' -P /content/ComfyUI/models/checkpoints/

# 3. ติดตั้ง Cloudflared
!wget -q -c https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

# 4. สตาร์ท Tunnel และดักจับ URL
def start_tunnel():
    proc = subprocess.Popen('cloudflared tunnel --url http://127.0.0.1:8188', shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if match:
            print('\n' + '='*65)
            print('🎉 ระบบพร้อมใช้งานแล้ว 100%! คลิกเปิด ComfyUI ที่ลิงก์ด้านล่างนี้:')
            print(f'👉 {match.group(0)}')
            print('='*65 + '\n')

threading.Thread(target=start_tunnel, daemon=True).start()
time.sleep(2)

%cd /content/ComfyUI
# รันด้วย Legacy Frontend + ปลดล็อก CORS Security (แก้ปัญหา 403 และแก้ค้างหน้า Splash Screen 100%)
!python main.py --listen 127.0.0.1 --port 8188 --enable-cors-header '*' --front-end-version Comfy-Org/ComfyUI_legacy_frontend@latest
